In [1]:
import os
import pickle

DATA_DIR = 'resources'
# Data dir = (1) working directory, (2) move out of test, (3) move out of src, and append to resources
DATA_DIR = os.path.join(os.getcwd(), '..', 'resources')
print("Data Directory:", DATA_DIR)

data = pickle.load(open(os.path.join(DATA_DIR, 'classified_book_dataset_6300.pickle'), 'rb'))
data_keys = list(data.keys())
print(f"Loaded {len(data)} records from the dataset.")

Data Directory: d:\claimpkg\claimpkg-clone\src\notebooks\..\resources
Loaded 6300 records from the dataset.


In [2]:
data_keys = data_keys[:5000]
data = {key: data[key] for key in data_keys}

# Count the number of easy, medium, and hard problems
counts = {'skipped': 0, 'easy': 0, 'medium': 0, 'hard': 0}
for key in data_keys:
    difficulty = data[key]['complexity']
    counts[difficulty] += 1
print("Counts of problems by difficulty:", counts)\

print("Number of problems that needs to be reprocessed:", counts['medium'] + counts['hard'])

Counts of problems by difficulty: {'skipped': 1780, 'easy': 1844, 'medium': 94, 'hard': 1282}
Number of problems that needs to be reprocessed: 1376


In [3]:
API_TEST_SET = [6, 8, 11]

DESIRE_OUTPUT = {
    0: """<e>unknown_0</e> || publisher || <e>Lippincott_Williams_&_Wilkins</e>;
<e>unknown_0</e> || country || <e>United_Kingdom</e>;
<e>United_Kingdom</e> || language || <e>English_language</e>""",
    6:"""<e>unknown_0</e> || deathPlace || <e>United_States</e>;
<e>unknown_0</e> || birthPlace || <e>Veckholm</e>""",
    8: """<e>Aenir</e> || precededBy || <e>Castle_(novel)</e>;
<e>Aenir</e> || subsequentWork || <e>unknown_0</e>;
<e>Into_Battle_(novel)</e> || previousWork || <e>unknown_0</e>""",
    11: """<e>Robert_Muldoon</e> || monarch || <e>Juan_Carlos_I_of_Spain</e>;
<e>Robert_Muldoon</e> || primeminister || <e>Felipe_González</e>;
<e>Felipe_González</e> || predecessor || <e>Wang_Xiankui</e>"""
}

In [4]:
# Testing the dataset
sample_key = data_keys[0]
print("Complexity of Sample Key:", data[sample_key]['complexity'], '\n')
print("Sample Key:", sample_key, '\n')
print("Sample Data:", data[sample_key]['Entity_set'], '\n')

print("Sample Claim Data:", data[sample_key]['Evidence'], '\n')

Complexity of Sample Key: easy 

Sample Key: is published by Lippincott Williams & Wilkins in the UK where English is the main language. 

Sample Data: ['United_Kingdom', 'English_language', 'Lippincott_Williams_&_Wilkins', 'AIDS_(journal)'] 

Sample Claim Data: {'United_Kingdom': [['language'], ['~country']], 'AIDS_(journal)': [['country'], ['publisher']], 'English_language': [['~language']], 'Lippincott_Williams_&_Wilkins': [['~publisher']]} 



In [5]:
import os
import sys
sys.path.append('..')
from llm.pseudograph_relabelling_llm import PseudoGraphRelabellingLLM

llm_processor = PseudoGraphRelabellingLLM()

outputs = []
for idx in API_TEST_SET:
    sample_key = data_keys[idx]
    sample_data = data[sample_key]
    print(f"Processing \"{sample_key}\"")

    output = llm_processor.submit(data_keys[idx], sample_data)
    outputs.append(output)

Processing "Well a person died in the United States and was born in Veckholm."
Processing "These novels were published in the order; Castle, Aenir, book and Into Battle."


Processing "Robert Muldoon, was in office during the reign of Monarch, Juan Carlos I of Spain and when the Prime Minister was Felipe González, whose predecessor was, Wang Xiankui."


In [6]:
import re
def extract_triplets(output: str) -> str:
    pattern = r"<e>.*?<\/e>\s*\|\|\s*.*?\s*\|\|\s*<e>.*?<\/e>"
    triplets = re.findall(pattern, output)
    return ';\n'.join(triplets)

for res in outputs:
    print(extract_triplets(res), '\n')

<e>unknown_0</e> || deathPlace || <e>United_States</e>;
<e>unknown_0</e> || birthPlace || <e>Veckholm</e> 

<e>Aenir</e> || precededBy || <e>Castle_(novel)</e>;
<e>Aenir</e> || subsequentWork || <e>unknown_0</e>;
<e>Into_Battle_(novel)</e> || previousWork || <e>unknown_0</e> 

<e>Robert_Muldoon</e> || monarch || <e>Juan_Carlos_I_of_Spain</e>;
<e>Robert_Muldoon</e> || primeminister || <e>Felipe_González</e>;
<e>Felipe_González</e> || predecessor || <e>Wang_Xiankui</e> 



The LLM looks good, now we find keys that need to relabelling


In [7]:
queue_to_relabel = []
for i, key in enumerate(data_keys):
    # If the complexity is medium or hard, add to the relabelling queue
    if data[key]['complexity'] in ['medium', 'hard']:
        queue_to_relabel.append(key)
print(f"Total number of problems to relabel: {len(queue_to_relabel)}")

Total number of problems to relabel: 1376


In [8]:
# Export the queue to a pickle file
import pickle
with open(os.path.join(DATA_DIR, 'relabel_queue.pickle'), 'wb') as f:
    pickle.dump(queue_to_relabel, f)